[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/antfolk/BMEN35_2026/blob/main/Session0/BMEN35_Ex0_04_Pandas_and_sklearn.ipynb)

# Pandas and one simple scikit-learn baseline

Now we will use Pandas to inspect a table and scikit-learn to make one very simple baseline prediction. We will **not** fit a more advanced model in this walkthrough. You will write that kind of model code yourself in the later exercises.

The dataset is the diabetes dataset included with scikit-learn. It contains 442 rows, ten baseline measurements, and a numerical measure of disease progression one year later. The feature values have been standardized, so they are not all in their original measurement units. This small dataset is for learning the workflow, not for clinical prediction.

Let's look at the table, plot one feature against the target, and then see what a `DummyRegressor` predicts. Run the cells in order and check what the baseline can and cannot tell us.

## 1. Load the table

The dataset loader can return a Pandas DataFrame. A DataFrame is a table with named columns and row indices. We will keep the feature table in `features` and the value we want to predict in `target`.

The only estimator we fit in this walkthrough is `DummyRegressor`. We also use scikit-learn helpers to load the example dataset, split the rows, and calculate an error metric. You will write code for more useful models yourself in the later exercises.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_diabetes

data = load_diabetes(as_frame=True)
features = data.data
target = data.target

print('Number of rows:', features.shape[0])
print('Number of feature columns:', features.shape[1])
print('Number of target values:', len(target))
print('Feature names:')
print(list(features.columns))

## 2. Look at the rows and columns

`head()` shows the first few rows. `info()` tells us how many values are present and what type each column has. `describe()` gives summary statistics, such as the mean and quartiles.

These are useful first checks. They do not tell us everything about the way the data were collected, but they help us catch obvious issues and understand the table before modeling.

In [ ]:
# Show the first five observations.
display(features.head())

# Check column types and non-missing value counts.
features.info()

# Compare the ranges and summaries of the feature columns.
display(features.describe().round(2).T)

# Count missing values in each feature column.
print('Missing values per feature:')
display(features.isna().sum().to_frame('missing values'))

The `bmi` column is a standardized feature, not BMI in the usual units. The target is a numerical follow-up measure. This is why we should read the dataset description before giving a feature a physical or clinical interpretation.

## 3. Plot one feature and the target

A table is useful, but let's plot some of the data too. We will put the `bmi` feature on the x-axis and the target on the y-axis. Each dot is one observation.

We only show one feature here. The full dataset has ten feature columns, and the baseline below will not use any of them.

In [ ]:
plt.scatter(features['bmi'], target, alpha=0.5, color='teal')
plt.xlabel('BMI feature (standardized values)')
plt.ylabel('One-year progression measure')
plt.title('One feature and the target')
plt.grid(alpha=0.25)
plt.show()

Look at the spread of points. A single feature does not determine one exact target value for every row. The plot also makes it clear that the feature values have been centered around zero, which is consistent with the dataset description that the measurements are standardized.

## 4. Set aside some observations

We will split the rows into a training set and a test set. The DummyRegressor will learn its one prediction from the training target values. We then check that prediction on the test set.

This is the same split idea you will use later when you build actual models. For now, our only model is the dummy baseline.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.25,
    random_state=42
)

print('Training rows:', len(X_train))
print('Test rows:', len(X_test))

## 5. Fit a DummyRegressor

A dummy regressor ignores the feature columns. With `strategy='mean'`, it predicts the average target value from the training set for every test observation.

This may seem too simple to be useful. That is exactly why it is a baseline: a model with more code should be compared with a simple reference. If it cannot do better than the baseline on suitable held-out data, we should ask what the extra model has learned.

In [ ]:
from sklearn.dummy import DummyRegressor

# This is the only prediction model in this walkthrough.
baseline = DummyRegressor(strategy='mean')

# Fit the baseline using the training target values.
baseline.fit(X_train, y_train)

# The feature rows are passed to predict, but this strategy ignores their values.
baseline_predictions = baseline.predict(X_test)

print('Mean target in training data:', round(y_train.mean(), 2))
print('First five baseline predictions:', baseline_predictions[:5])

All five predictions are the same. That is not a bug: `strategy='mean'` always predicts the training-set mean. The feature values were passed to the `predict` method, but the dummy strategy does not use them.

## 6. Measure the baseline error

Mean absolute error (MAE) is the average size of the prediction error, ignoring whether the prediction was too high or too low. For a baseline, this tells us how far the constant prediction is from the test targets on average.

We will calculate MAE for the DummyRegressor and also write out the calculation with NumPy. These two values should agree, apart from small rounding differences.

In [ ]:
from sklearn.metrics import mean_absolute_error

baseline_mae = mean_absolute_error(y_test, baseline_predictions)
print('DummyRegressor MAE:', round(baseline_mae, 2))

# Calculate the absolute errors one by one, then take their average.
absolute_errors = np.abs(y_test.to_numpy() - baseline_predictions)
manual_mae = absolute_errors.sum() / len(absolute_errors)
print('MAE calculated with NumPy:', round(manual_mae, 2))
print('First five absolute errors:', absolute_errors[:5])

## 7. Plot the baseline predictions

Let's plot observed target values on the horizontal axis and predictions on the vertical axis. A perfect prediction would fall on the diagonal. Because this baseline always predicts the same mean, its points lie along a horizontal line.

The plot shows something that one MAE number cannot: this model makes the same prediction for every observation, no matter what the features are.

In [ ]:
low = min(y_test.min(), baseline_predictions.min())
high = max(y_test.max(), baseline_predictions.max())

plt.figure(figsize=(6, 6))
plt.scatter(y_test, baseline_predictions, alpha=0.6, color='darkorange')
plt.plot([low, high], [low, high], '--', color='black', label='perfect prediction')
plt.xlabel('Observed target')
plt.ylabel('DummyRegressor prediction')
plt.title('One constant prediction for every test observation')
plt.legend()
plt.grid(alpha=0.25)
plt.show()

## Pause and check

1. What number does `DummyRegressor(strategy='mean')` use for every prediction?
2. Does it look at the feature values? How can you tell from the printed predictions?
3. What does the MAE tell us about the errors? What does it not tell us?
4. If you later fit a more complicated model, what would you compare it with?
5. Why is this exercise not evidence that a model can predict an individual's health outcome?

This walkthrough has shown how to inspect a Pandas table, make a plot, and establish one simple reference prediction with scikit-learn. In later exercises you will write the code for the actual models and compare them with an appropriate baseline.